In [3]:
from astropy.cosmology import Planck18
%env XLA_PYTHON_CLIENT_ALLOCATOR=platform
import astropy.units as u
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
from glob import glob
import numpy as np
import sys
sys.path.append('../src/')
import jax.numpy as jnp
import matplotlib.pyplot as plt
from gwosc.api import fetch_event_json
import re
import os
import jax
import h5py
import pandas as pd
jax.local_devices()
jax.config.update("jax_enable_x64", True)


env: XLA_PYTHON_CLIENT_ALLOCATOR=platform


In [4]:

def get_event_far(file):
    match = re.search(r"(GW\d{6}_\d{6})", os.path.basename(file))
    if not match:
        return (10, None)

    event = match.group(1)

    try:
        evt_dict = fetch_event_json(event)
    except Exception as e:
        print(f"Failed to fetch {event}: {e}")
        return (10, event)

    try:
        evt_name = list(evt_dict["events"].keys())[0]
        data = evt_dict["events"][evt_name]
    except Exception as e:
        print(f"Malformed response for {event}: {e}")
        return (10, event)

    # Extract FAR
    far = data.get("far", data.get("far"))
    #far = 1 / (far_Hz * seconds_per_year)
    if far is None:
        print(f"No FAR found for {event}. Keys: {list(data.keys())}")
        return (10, event)

    return far, event

In [ ]:
#data_paths = ['/mnt/home/ccalvk/ceph/GWTC-4/IGWN-GWTC4p0-*-combined_PEDataRelease.hdf5',]
#data_paths=['/mnt/home/ccalvk/ceph/GWTC-3/IGWN-GWTC3p0-v2-GW*_PEDataRelease_mixed_nocosmo.h5']
data_paths=['/mnt/home/misi/ceph/rp.04/catalogs/GWTC-5/GWTC5-Stable_Release-8/4c51f4201_25/bbh_only/IGWN-GWTC5p0-*-combined_PEDataRelease.hdf5']
files = []
for path in data_paths:
    files += glob(path)

print(len(files))

104


In [6]:
import os
import re
from gwosc.api import fetch_event_json

def get_event_far(file):
    match = re.search(r"(GW\d{6}_\d{6})", os.path.basename(file))
    if not match:
        return None, None

    event = match.group(1)

    try:
        evt_dict = fetch_event_json(event)

        evt_name = next(iter(evt_dict["events"]))
        far = evt_dict["events"][evt_name].get("far")

        return far, event

    except Exception as e:
        print(f"Failed {event}: {e}")
        return None, event

In [32]:
fars = []
names = []
for file in files:
    far, event = get_event_far(file)
    if far is None:
        continue
    if far < 1:
        fars.append(far)
        names.append(event)

Failed GW241009_084816: HTTPSConnectionPool(host='gwosc.org', port=443): Read timed out. (read timeout=10)


In [35]:
len(names)

103

In [ ]:
event_ifars=pd.read_csv('/mnt/home/misi/src/Memory/results/prod_20260402c/auto_o3o4a_joint/event_ifars.txt', sep=' ')
event_ifars['far']=1/event_ifars['event_name']
print(len(np.where(event_ifars['far']<1)[0]))
events_TC4=event_ifars.loc[event_ifars['#'].str.startswith('GW2')]
good_evt_TC4=events_TC4.iloc[np.where(events_TC4['far']<1)]

148


In [136]:
only_TC4 = set(good_evt_TC4['#']).difference(names)
print((only_TC4)) #all just GWTC3 as it should be

{'GW200302_015811', 'GW200209_085452', 'GW200316_215756', 'GW200202_154313', 'GW200112_155838', 'GW200224_222234', 'GW200129_065458', 'GW200128_022011', 'GW200225_060421', 'GW200208_130117', 'GW200216_220804', 'GW200311_115853', 'GW200219_094415'}


In [137]:
not_TC4 = set(names).difference(good_evt_TC4['#'])
print((not_TC4)) # two low mass ones i need to cut

{'GW230529_181500', 'GW230518_125908'}


In [ ]:
far_threshold=1
include=[]
fars=[]
for i, file in enumerate(files):
    samples=[]
    event_far, name = get_event_far(file)
    fars.append(event_far)

    with h5py.File(file, 'r') as f:
        if 'PublicationSamples' in f.keys():
            # O3a files
            samples = np.array(f['PublicationSamples']['posterior_samples'])
        elif 'C00:Mixed' in f.keys():
            # O3b files
            samples = np.array(f['C00:Mixed']['posterior_samples'])
        elif 'C00:NRSur7dq4' in f.keys(): #what waveform approximation did we use
            samples = np.array(f['C00:NRSur7dq4']['posterior_samples'])        
        elif 'C00:IMRPhenomNSBH' in f.keys(): #what waveform approximation did we use
            samples = np.array(f['C00:IMRPhenomNSBH']['posterior_samples'])   
        elif 'C00:IMRPhenomNSBH:LowSpin' in f.keys(): #what waveform approximation did we use
            samples = np.array(f['C00:IMRPhenomNSBH:LowSpin']['posterior_samples'])     
        elif 'C01:IMRPhenomXPHM-SpinTaylor' in f.keys(): # GWTC5
            samples = np.array(f['C01:IMRPhenomXPHM-SpinTaylor']['posterior_samples'])   
        elif 'C00:IMRPhenomXPHM-SpinTaylor' in f.keys(): # GWTC5
            samples = np.array(f['C00:IMRPhenomXPHM-SpinTaylor']['posterior_samples'])        
        else:   
            print(f"Available keys in file {name}: {list(f.keys())}")
            continue
    zs=samples['redshift'] [()]
    m1_det = samples['mass_1'][()]
    qs = samples['mass_ratio'][()]
    dLs = samples['luminosity_distance'][()] / 1e3

    # should this be reweighted? and if so, whats the prior here, bc this is combined not nocosmo
    m2s_det= m1_det * qs
    m2s_src= m2s_det / (1 + zs)
    prob = np.mean(m2s_src > 2.5)
    #df_det_chunk['m1d'] = df_det_chunk['m1'] * (1 + df_det_chunk['z']) 

    if prob !=1:
        print(name, prob)
    if event_far < far_threshold and prob>0.9:
        include.append(name)


GW241109_115924 0.9997256892058702


In [41]:
len(include)

104

In [ ]:
file1 = open("../runs/INCLUDE_LIST.txt", "a")
for name in include:
    file1.write(name + "\n")
file1.close()

In [12]:
INCLUDE_LIST=[]
with open("../runs/INCLUDE_LIST.txt", "r") as f:
    INCLUDE_LIST = set(line.strip() for line in f if line.strip())
    filtered_files = []
for f in files:
    filename = os.path.basename(f)
    parts = re.split("_|-", filename)
    if len(parts) >= 2 and parts[1] != 'GWTC4p0':
        event_name = parts[3] + "_" + parts[4]
        if event_name in INCLUDE_LIST:
            filtered_files.append(f)
    if len(parts) >= 2 and parts[1] == 'GWTC4p0' or parts[1] == 'GWTC5p0':
        event_name = parts[4] + "_" + parts[5]
        if event_name in INCLUDE_LIST:
            filtered_files.append(f)
    print(event_name)

GW240925_005809
GW241002_030559
GW240908_082628
GW241231_054133
GW241210_120900
GW250114_082203
GW241230_084504
GW240630_101703
GW250118_023225
GW241009_022835
GW240716_034900
GW241124_024914
GW240601_061200
GW240420_175625
GW240527_183429
GW240612_081540
GW240615_113620
GW250119_025138
GW250118_055802
GW240930_035959
GW240514_121713
GW240924_000316
GW240527_230910
GW240916_184352
GW240622_004008
GW240703_191355
GW240531_040326
GW240919_061559
GW240920_073424
GW240505_133552
GW240902_143306
GW240525_031210
GW240520_213616
GW240621_195059
GW250109_010541
GW241210_060606
GW240920_124024
GW241127_061008
GW241109_115924
GW240511_031507
GW240830_211120
GW240526_093944
GW240615_160735
GW250119_190238
GW241102_144729
GW240414_054515
GW240908_125134
GW241114_235258
GW250116_015318
GW241201_055758
GW241110_124123
GW241007_082943
GW250101_011205
GW241116_151753
GW240629_145256
GW240825_055146
GW250109_074552
GW240530_012417
GW250104_015122
GW241101_220523
GW240519_012815
GW240601_231004
GW241114

In [10]:
len(filtered_files)

103